### Load the libraries

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

In [3]:
### Load the Dataset

In [4]:
df = pd.read_csv(r'D:\CUSTOMER-CHURN\Dataset\churn_dataset_cleaned.csv')
df.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,0,Yes,No,1,No,No,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,Male,0,No,No,45,No,No,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [5]:
### Dataset overview

In [6]:
df.shape

(7043, 20)

In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 20 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   gender            7043 non-null   object 
 1   SeniorCitizen     7043 non-null   int64  
 2   Partner           7043 non-null   object 
 3   Dependents        7043 non-null   object 
 4   tenure            7043 non-null   int64  
 5   PhoneService      7043 non-null   object 
 6   MultipleLines     7043 non-null   object 
 7   InternetService   7043 non-null   object 
 8   OnlineSecurity    7043 non-null   object 
 9   OnlineBackup      7043 non-null   object 
 10  DeviceProtection  7043 non-null   object 
 11  TechSupport       7043 non-null   object 
 12  StreamingTV       7043 non-null   object 
 13  StreamingMovies   7043 non-null   object 
 14  Contract          7043 non-null   object 
 15  PaperlessBilling  7043 non-null   object 
 16  PaymentMethod     7043 non-null   object 


### Problem Statement
* Build a system which can take features of a Telecom Customer like tenure, MonthlyCharges, Contract type, InternetService, etc., and predict whether they will Churn.
* Target variable - Churn (Yes/No mapped to 1/0)
* Evaluation Metric - Accuracy Score


### Data Preprocessing

#### STEP 1: CLEANING, ENCODING & SPLITTING

In [27]:
# ==========================================
# STEP 1: CLEANING, ENCODING & SPLITTING
# ==========================================
import pandas as pd
from sklearn.model_selection import train_test_split

# 1. Load the dataset
df = pd.read_csv(r'D:\CUSTOMER-CHURN\Dataset\churn_dataset_cleaned.csv')

# 2. Drop customerID and strictly drop TotalCharges to prevent Multicollinearity
cols_to_drop = [col for col in ['customerID', 'TotalCharges'] if col in df.columns]
df.drop(columns=cols_to_drop, inplace=True, errors='ignore')

# 3. Encode Target Variable (Churn)
df['Churn'] = df['Churn'].replace({'Yes': 1, 'No': 0})

# 4. Separate Features (X) and Target (Y)
Y = df['Churn'] 
X = df.drop(columns=['Churn'])

# 5. Categorical Encoding (Dummy Variables)
X = pd.get_dummies(X, drop_first=True, dtype=int)

# 6. Train/Test Split (Stratified to keep the 73/27 realistic ratio in the test set)
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.25, random_state=42, stratify=Y)

# Save column layout for Streamlit deployment later
colos = X_train.columns.tolist()

print("✅ Step 1 Complete: Data Cleaned, Encoded, and Split!")
print(f"Shape of X_train: {X_train.shape}")
print(f"Shape of X_test: {X_test.shape}")

✅ Step 1 Complete: Data Cleaned, Encoded, and Split!
Shape of X_train: (5282, 22)
Shape of X_test: (1761, 22)


#### STEP 2: TARGETED SCALING

In [28]:
# ==========================================
# STEP 2: TARGETED SCALING
# ==========================================
from sklearn.preprocessing import StandardScaler

# 1. Initialize the Scaler
scale = StandardScaler()

# 2. Make copies to safely store our scaled data
X_train_transformed = X_train.copy()
X_test_transformed = X_test.copy()

# 3. Define the purely continuous columns
num_cols = ['tenure', 'MonthlyCharges']

# 4. Fit the scaler ONLY on the training data's numeric columns
X_train_transformed[num_cols] = scale.fit_transform(X_train[num_cols])

# 5. Transform the test data's numeric columns
X_test_transformed[num_cols] = scale.transform(X_test[num_cols])

print("✅ Step 2 Complete: Continuous Variables Scaled!")
display(X_train_transformed.head())

✅ Step 2 Complete: Continuous Variables Scaled!


,SeniorCitizen,tenure,MonthlyCharges,gender_Male,Partner_Yes,Dependents_Yes,PhoneService_Yes,MultipleLines_Yes,InternetService_Fiber optic,InternetService_No,...,DeviceProtection_Yes,TechSupport_Yes,StreamingTV_Yes,StreamingMovies_Yes,Contract_One year,Contract_Two year,PaperlessBilling_Yes,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
6661,0,1.609608,-0.371461,0,1,1,0,0,0,0,...,0,1,1,1,0,1,0,1,0,0
4811,0,-1.151780,-0.625032,0,0,0,1,0,0,0,...,0,0,0,0,0,0,1,0,0,1
2193,0,0.959870,-1.447065,1,0,1,1,0,0,1,...,0,0,0,0,1,0,1,0,0,1
1904,0,0.959870,0.980917,1,0,0,1,1,1,0,...,0,1,0,1,0,0,1,0,1,0
6667,0,-0.948737,0.487034,0,0,0,1,0,1,0,...,0,0,0,1,0,0,1,0,1,0


#### STEP 3: HANDLING IMBALANCE (SMOTE)

In [29]:
# ==========================================
# STEP 3: HANDLING IMBALANCE (SMOTE)
# ==========================================
from imblearn.over_sampling import SMOTE

# 1. Initialize SMOTE with a random state for reproducibility
smote = SMOTE(random_state=42)

# 2. Fit and apply SMOTE strictly to the transformed training data
X_train_balanced, Y_train_balanced = smote.fit_resample(X_train_transformed, Y_train)

# 3. Verify the balancing worked
print("✅ Step 3 Complete: Training Data Balanced!")
print("\n--- Before SMOTE (Imbalanced) ---")
print(Y_train.value_counts())

print("\n--- After SMOTE (Perfectly Balanced) ---")
print(Y_train_balanced.value_counts())

✅ Step 3 Complete: Training Data Balanced!

--- Before SMOTE (Imbalanced) ---
Churn
0    3880
1    1402
Name: count, dtype: int64

--- After SMOTE (Perfectly Balanced) ---
Churn
0    3880
1    3880
Name: count, dtype: int64


#### STEP 4: FAIR MODEL TRAINING & EVALUATION

In [30]:
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Initialize an empty list to store the results of every model
evaluation_results = []
print("✅ Setup complete. Ready to train models!")

✅ Setup complete. Ready to train models!


### 1️⃣ Logistic Regression (SMOTE)

In [31]:
from sklearn.linear_model import LogisticRegression

# Initialize and Train
lr_smote = LogisticRegression(max_iter=1000, random_state=42)
lr_smote.fit(X_train_balanced, Y_train_balanced)

# Predict and Evaluate
y_pred_lr = lr_smote.predict(X_test_transformed)

evaluation_results.append({
    'Algorithm': 'Logistic Regression (SMOTE)',
    'Accuracy': accuracy_score(Y_test, y_pred_lr),
    'Precision': precision_score(Y_test, y_pred_lr),
    'Recall': recall_score(Y_test, y_pred_lr),
    'F1-Score': f1_score(Y_test, y_pred_lr)
})
print("✅ Logistic Regression trained and evaluated!")

✅ Logistic Regression trained and evaluated!


#### 2️⃣ K-Nearest Neighbors - KNN (SMOTE)

In [32]:
from sklearn.neighbors import KNeighborsClassifier

# Initialize and Train
knn_smote = KNeighborsClassifier()
knn_smote.fit(X_train_balanced, Y_train_balanced)

# Predict and Evaluate
y_pred_knn = knn_smote.predict(X_test_transformed)

evaluation_results.append({
    'Algorithm': 'KNN (SMOTE)',
    'Accuracy': accuracy_score(Y_test, y_pred_knn),
    'Precision': precision_score(Y_test, y_pred_knn),
    'Recall': recall_score(Y_test, y_pred_knn),
    'F1-Score': f1_score(Y_test, y_pred_knn)
})
print("✅ KNN trained and evaluated!")

✅ KNN trained and evaluated!


### 3️⃣ Decision Tree (SMOTE)

In [33]:
from sklearn.tree import DecisionTreeClassifier

# Initialize and Train
dt_smote = DecisionTreeClassifier(random_state=42)
dt_smote.fit(X_train_balanced, Y_train_balanced)

# Predict and Evaluate
y_pred_dt = dt_smote.predict(X_test_transformed)

evaluation_results.append({
    'Algorithm': 'Decision Tree (SMOTE)',
    'Accuracy': accuracy_score(Y_test, y_pred_dt),
    'Precision': precision_score(Y_test, y_pred_dt),
    'Recall': recall_score(Y_test, y_pred_dt),
    'F1-Score': f1_score(Y_test, y_pred_dt)
})
print("✅ Decision Tree trained and evaluated!")

✅ Decision Tree trained and evaluated!


### 4️⃣ Random Forest (SMOTE)

In [34]:
from sklearn.ensemble import RandomForestClassifier

# Initialize and Train
rf_smote = RandomForestClassifier(random_state=42)
rf_smote.fit(X_train_balanced, Y_train_balanced)

# Predict and Evaluate
y_pred_rf = rf_smote.predict(X_test_transformed)

evaluation_results.append({
    'Algorithm': 'Random Forest (SMOTE)',
    'Accuracy': accuracy_score(Y_test, y_pred_rf),
    'Precision': precision_score(Y_test, y_pred_rf),
    'Recall': recall_score(Y_test, y_pred_rf),
    'F1-Score': f1_score(Y_test, y_pred_rf)
})
print("✅ Random Forest trained and evaluated!")

✅ Random Forest trained and evaluated!


### 5️⃣ Gradient Boosting - GBDT (SMOTE)

In [35]:
from sklearn.ensemble import GradientBoostingClassifier

# Initialize and Train
gbdt_smote = GradientBoostingClassifier(random_state=42)
gbdt_smote.fit(X_train_balanced, Y_train_balanced)

# Predict and Evaluate
y_pred_gbdt = gbdt_smote.predict(X_test_transformed)

evaluation_results.append({
    'Algorithm': 'Gradient Boosting (SMOTE)',
    'Accuracy': accuracy_score(Y_test, y_pred_gbdt),
    'Precision': precision_score(Y_test, y_pred_gbdt),
    'Recall': recall_score(Y_test, y_pred_gbdt),
    'F1-Score': f1_score(Y_test, y_pred_gbdt)
})
print("✅ GBDT trained and evaluated!")

✅ GBDT trained and evaluated!


### Evaluation

In [36]:
# Convert our list of dictionaries into a DataFrame
final_leaderboard = pd.DataFrame(evaluation_results)

# Sort by F1-Score (Highest to Lowest)
final_leaderboard.sort_values(by='F1-Score', ascending=False, inplace=True)
final_leaderboard.reset_index(drop=True, inplace=True)

print("🏆 The Ultimate SMOTE Leaderboard 🏆")
display(final_leaderboard)

🏆 The Ultimate SMOTE Leaderboard 🏆


,Algorithm,Accuracy,Precision,Recall,F1-Score
0,Gradient Boosting (SMOTE),0.763771,0.538231,0.768737,0.633157
1,Logistic Regression (SMOTE),0.764338,0.540625,0.740899,0.625113
2,KNN (SMOTE),0.723453,0.485955,0.740899,0.586938
3,Random Forest (SMOTE),0.769449,0.560878,0.601713,0.580579
4,Decision Tree (SMOTE),0.708688,0.459220,0.554604,0.502425


In [38]:
# ==========================================
# STEP 5: HYPERPARAMETER TUNING (GBDT)
# ==========================================
from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pandas as pd

print("Hunting for the best hyperparameters... (This might take 1-2 minutes)")

# 1. Define the base model
gbdt_base = GradientBoostingClassifier(random_state=42)

# 2. Define the grid of parameters to test
param_grid = {
    'n_estimators': [100, 200, 300],          # How many trees to build
    'learning_rate': [0.01, 0.05, 0.1, 0.2],  # How aggressively the trees correct errors
    'max_depth': [3, 4, 5],                   # Restricting depth prevents overfitting
    'min_samples_split': [2, 5, 10]           # Minimum samples needed to split a node
}

# 3. Setup RandomizedSearchCV (Tests 20 random combinations using 3-fold cross-validation)
random_search = RandomizedSearchCV(
    estimator=gbdt_base, 
    param_distributions=param_grid, 
    n_iter=20, 
    cv=3, 
    scoring='f1', # We optimize specifically for F1-Score!
    n_jobs=-1,    # Uses all your computer's CPU cores to run faster
    random_state=42
)

# 4. Train the search strictly on the BALANCED data
random_search.fit(X_train_balanced, Y_train_balanced)

# 5. Extract the absolute best model
best_gbdt_tuned = random_search.best_estimator_

print("\n✅ Tuning Complete!")
print("Best Parameters Found:")
print(random_search.best_params_)

# 6. Predict and Evaluate the Tuned Model on UNSEEN test data
y_pred_tuned = best_gbdt_tuned.predict(X_test_transformed)

acc_tuned = accuracy_score(Y_test, y_pred_tuned)
prec_tuned = precision_score(Y_test, y_pred_tuned)
rec_tuned = recall_score(Y_test, y_pred_tuned)
f1_tuned = f1_score(Y_test, y_pred_tuned)

# 7. Add it to our final leaderboard to compare
tuned_row = pd.DataFrame({
    'Algorithm': ['TUNED Gradient Boosting (SMOTE)'],
    'Accuracy': [acc_tuned],
    'Precision': [prec_tuned],
    'Recall': [rec_tuned],
    'F1-Score': [f1_tuned]
})

final_leaderboard = pd.concat([final_leaderboard, tuned_row], ignore_index=True)
final_leaderboard.sort_values(by='F1-Score', ascending=False, inplace=True)
final_leaderboard.reset_index(drop=True, inplace=True)

print("\n🏆 The Ultimate Tuned Leaderboard 🏆")
display(final_leaderboard)

Hunting for the best hyperparameters... (This might take 1-2 minutes)

✅ Tuning Complete!
Best Parameters Found:
{'n_estimators': 100, 'min_samples_split': 2, 'max_depth': 5, 'learning_rate': 0.05}

🏆 The Ultimate Tuned Leaderboard 🏆


,Algorithm,Accuracy,Precision,Recall,F1-Score
0,Gradient Boosting (SMOTE),0.763771,0.538231,0.768737,0.633157
1,Logistic Regression (SMOTE),0.764338,0.540625,0.740899,0.625113
2,🔥 TUNED Gradient Boosting (SMOTE),0.756388,0.529412,0.732334,0.614555
3,TUNED Gradient Boosting (SMOTE),0.756388,0.529412,0.732334,0.614555
4,KNN (SMOTE),0.723453,0.485955,0.740899,0.586938
5,Random Forest (SMOTE),0.769449,0.560878,0.601713,0.580579
6,Decision Tree (SMOTE),0.708688,0.459220,0.554604,0.502425
